#### Area Points

###### Importing Libraires and Loading Data

In [1]:
from pathlib import Path
import pandas as pd
import geopandas as gpd

In [2]:
project_root = Path.cwd().parent
raw = project_root/'data'/'raw'
out_dir = project_root/'data'/'processed'/'area'
bng = 'EPSG:27700'

In [3]:
# GP practices only have a postcode, so this gives them coordinates

pc = pd.read_parquet(out_dir/'postcodes.parquet', columns=['postcode', 'easting', 'northing'])

In [4]:
# Each section below appends one small table here and they are stacked into one at the end

parts = []

##### Schools

In [5]:
# Get Information About Schools (GIAS): the official register of every school in England

gias = pd.read_csv(raw/'schools'/'schools_register_gias_2026_10_14.csv', low_memory=False)

In [6]:
# Keep open schools of the kinds parents mean by "school". Universities, colleges and
# other types are left out

keep_groups = ['Local authority maintained schools', 'Academies', 'Free Schools', 'Special schools', 'Independent schools']
schools = gias[(gias['EstablishmentStatus (name)'] == 'Open') & gias['EstablishmentTypeGroup (name)'].isin(keep_groups)]

In [7]:
# 7 schools have no coordinates and a few have 0 so they can't be placed on a map, so we delete them

schools = schools.dropna(subset=['Easting', 'Northing'])
schools = schools[schools['Easting'] > 0]

In [8]:
print(schools.shape)
print(schools['EstablishmentTypeGroup (name)'].value_counts())

(3065, 135)
EstablishmentTypeGroup (name)
Local authority maintained schools    1369
Academies                              913
Independent schools                    420
Free Schools                           193
Special schools                        170
Name: count, dtype: int64


In [10]:
# Classify each school by the ages it takes. GIAS's own "phase" is "Not applicable" for most
# independent and special schools, so one rule based on ages treats every school the same way

def school_level(low, high):
    if high <= 5:
        return 'nursery'
    if low >= 16:
        return 'sixth form'
    if high <= 13:
        return 'primary'
    if low >= 9:
        return 'secondary'
    return 'all-through'

In [11]:
schools = schools.assign(level=[school_level(lo, hi) for lo, hi in zip(schools['StatutoryLowAge'], schools['StatutoryHighAge'])])


In [12]:
print(pd.crosstab(schools['PhaseOfEducation (name)'], schools['level']))

level                    all-through  nursery  primary  secondary  sixth form
PhaseOfEducation (name)                                                      
16 plus                            0        0        0          0          17
All-through                       42        0        0          0           0
Not applicable                   264        2      255        200           0
Nursery                            0       74        0          0           0
Primary                            0        0     1752          0           0
Secondary                          1        0        0        458           0


In [13]:
# Office of Standards in Education (Ofsted) latest inspection results for every state-funded school

ofsted = pd.read_csv(raw/'schools'/'schools_ofsted_latest_inspections_2026_08_31.csv', dtype=str)

In [14]:
# Pre-September-2024 grade numbers, and the six areas of the new report cards (since November 2025)

grade = {'1': 'Outstanding', '2': 'Good', '3': 'Requires improvement', '4': 'Inadequate'}
report_cols = ['Inclusion', 'Curriculum and teaching', 'Achievement', 'Attendance and behaviour',
               'Personal development and wellbeing', 'Leadership and governance']

In [15]:
# One readable summary per school, listing every inspection type it has, each with its date

def ofsted_summary(r):
    found = []
    if pd.notna(r['Inclusion']):
        grades = ', '.join(f'{c}: {r[c]}' for c in report_cols)
        found.append(f"Report card ({r['Inspection start date']}): {grades}, Safeguarding: {r['Safeguarding standards']}")
    overall = r['Latest OEIF overall effectiveness']
    if pd.notna(overall):
        quality = str(r['Latest OEIF quality of education']).split('.')[0]
        found.append(f"Graded inspection ({r['Inspection start date of latest OEIF graded inspection']}): "
                     f"{grade.get(overall, overall)} overall, quality of education {grade.get(quality, quality)}")
    if pd.notna(r['Ungraded inspection overall outcome']):
        found.append(f"Ungraded inspection ({r['Date of latest ungraded inspection']}): {r['Ungraded inspection overall outcome']}")
    return ' | '.join(found) or 'Not yet inspected'

In [17]:
ofsted['ofsted'] = ofsted.apply(ofsted_summary, axis=1)
ofsted['URN'] = ofsted['URN'].astype(int)
print(ofsted['ofsted'].value_counts().head(5))
for s in ofsted['ofsted'].sample(3, random_state=1):
    print('-', s)

ofsted
Not yet inspected                                                          30
Graded inspection (11/07/2023): Good overall, quality of education Good    15
Graded inspection (20/06/2023): Good overall, quality of education Good    14
Ungraded inspection (16/10/2024): Standards maintained                     14
Ungraded inspection (18/12/2024): Standards maintained                     13
Name: count, dtype: int64
- Graded inspection (06/02/2024): Requires improvement overall, quality of education Requires improvement
- Ungraded inspection (16/10/2024): Standards maintained
- Report card (12/05/2026): Inclusion: Strong standard, Curriculum and teaching: Strong standard, Achievement: Strong standard, Attendance and behaviour: Strong standard, Personal development and wellbeing: Strong standard, Leadership and governance: Strong standard, Safeguarding: Met


In [18]:
# Attach each school's Ofsted summary using URN, the school's unique ID in both files

schools = schools.merge(ofsted[['URN', 'ofsted']], on='URN', how='left')
schools['ofsted'] = schools['ofsted'].fillna('No state-school Ofsted record')

In [19]:
# One readable sentence per school for the chatbot, e.g.
# 'Primary, Academy converter, ages 4-11, 420 pupils, Church of England. Ofsted: ...'
# Pieces that don't apply become None and are skipped

def school_info(r):
    info = [r['level'].capitalize(), r['TypeOfEstablishment (name)'],
            f"ages {int(r['StatutoryLowAge'])}-{int(r['StatutoryHighAge'])}" if pd.notna(r['StatutoryLowAge']) else None,
            f"{int(r['NumberOfPupils'])} pupils" if pd.notna(r['NumberOfPupils']) else None,
            r['Gender (name)'] if r['Gender (name)'] in ['Girls', 'Boys'] else None,
            r['ReligiousCharacter (name)'] if r['ReligiousCharacter (name)'] not in ['Does not apply', 'None'] else None]
    return ', '.join(i for i in info if isinstance(i, str)) + '. Ofsted: ' + r['ofsted']

In [20]:
# Schools in the shared points format

parts.append(pd.DataFrame({
    'type': 'school', 'subtype': schools['level'], 'id': schools['URN'].astype(str),
    'name': schools['EstablishmentName'], 'info': schools.apply(school_info, axis=1),
    'easting': schools['Easting'], 'northing': schools['Northing'],
}))

In [21]:
print(parts[-1].shape)
print(parts[-1]['info'].iloc[0])

(3065, 7)
Primary, Voluntary aided school, ages 3-11, 229 pupils, Church of England. Ofsted: Graded inspection (12/06/2024): Outstanding overall, quality of education Outstanding


##### GP Practices

In [23]:
# NHS England's list of GP practices and other prescribing centres in the London region

gp = pd.read_csv(raw/'health'/'gp_practices_nhs_2026_10.csv', dtype=str)

In [24]:
# Keep practices that are open (ACTIVE) and are actual GP surgeries (role code RO76);
# the other codes are walk-in centres, out-of-hours services, hospital clinics, etc.

gp = gp[(gp['Status Code'] == 'ACTIVE') & (gp['Prescribing Setting (ODS role code)'] == 'RO76')]

In [25]:
# The file has a postcode but no coordinates, so take them from the postcode table

gp = gp.merge(pc, left_on='Postcode', right_on='postcode', how='inner')

In [26]:
print(gp.shape)

(1142, 30)


In [27]:
# GP practices in the shared points format; info = the address on one line

addr_cols = ['Address Line 1', 'Address Line 2', 'Address Line 3', 'Address Line 4', 'Address Line 5', 'Postcode']
parts.append(pd.DataFrame({
    'type': 'gp', 'subtype': 'gp practice', 'id': gp['Organisation Code'],
    'name': gp['Name'], 'info': gp[addr_cols].apply(lambda r: ', '.join(r.dropna()), axis=1),
    'easting': gp['easting'], 'northing': gp['northing'],
}))

In [29]:
print(parts[-1].head(5))

  type      subtype      id                           name  \
0   gp  gp practice  E83003    OAKLEIGH ROAD HEALTH CENTRE   
1   gp  gp practice  E83005        LICHFIELD GROVE SURGERY   
2   gp  gp practice  E83006      GREENFIELD MEDICAL CENTRE   
3   gp  gp practice  E83007  SQUIRES LANE MEDICAL PRACTICE   
4   gp  gp practice  E83008     HEATHFIELDE MEDICAL CENTRE   

                                                info   easting  northing  
0  280 OAKLEIGH ROAD NORTH, WHETSTONE, LONDON, N2...  527545.0  193427.0  
1       64 LICHFIELD GROVE, FINCHLEY, LONDON, N3 2JP  525236.0  190515.0  
2          143-145 CRICKLEWOOD LANE, LONDON, NW2 1HS  524477.0  186207.0  
3           2 SQUIRES LANE, FINCHLEY, LONDON, N3 2AU  526493.0  190872.0  
4   LYTTELTON ROAD, FINCHLEY, BARNET, LONDON, N2 0EQ  526207.0  188945.0  


##### Supermarkets

In [30]:
# GEOLYTIX Retail Points: every grocery store in London with its chain and size band

shops = pd.read_csv(raw/'supermarkets'/'supermarkets_geolytix_2026_06.csv')

In [31]:
print(shops.shape)
print(shops['size_band'].value_counts())

(2059, 17)
size_band
< 3,013 ft2 (280m2)                       1309
3,013 < 15,069 ft2 (280 < 1,400 m2)        473
30,138 ft2 > (2,800 m2)                    143
15,069 < 30,138 ft2 (1,400 < 2,800 m2)     134
Name: count, dtype: int64


In [32]:
# subtype = the chain, so the chatbot can answer "nearest Lidl" as well as "nearest supermarket"

parts.append(pd.DataFrame({
    'type': 'supermarket', 'subtype': shops['retailer'], 'id': shops['id'].astype(str),
    'name': shops['store_name'], 'info': shops['retailer'] + ', ' + shops['size_band'],
    'easting': shops['bng_e'], 'northing': shops['bng_n'],
}))

##### Parks and Playgrounds

In [33]:
# OS Open Greenspace: site outlines (with their type and name) and their entrances
# Entrances matter because a big park's centre can be far from any gate

sites = gpd.read_file(raw/'greenspace'/'greenspace_sites_polygons_2026_03.geojson')
entrances = gpd.read_file(raw/'greenspace'/'greenspace_entrances_points_2026_03.geojson').to_crs(bng)

In [34]:
print(sites['function'].value_counts())

function
Play Space                                3531
Public Park Or Garden                     1941
Other Sports Facility                     1683
Tennis Court                              1082
Religious Grounds                          794
Allotments Or Community Growing Spaces     780
Playing Field                              762
Bowling Green                              284
Cemetery                                   215
Golf Course                                132
Name: count, dtype: int64


In [35]:
# Keep the green spaces people mean by "park", with friendlier subtype names

park_types = {'Public Park Or Garden': 'park', 'Play Space': 'playground',
              'Playing Field': 'playing field', 'Allotments Or Community Growing Spaces': 'allotment'}
sites = sites[sites['function'].isin(park_types)]

In [36]:
# Give each entrance its site's type and name.

parks = entrances.merge(sites[['id', 'function', 'distName1']], left_on='refToGSite', right_on='id', suffixes=('_entrance', ''))
print(parks.shape, parks['id'].nunique(), 'sites')

(18123, 7) 6713 sites


In [37]:
# Parks in the shared points format. id = the SITE id, so a park with 6 gates has 6 rows
# with the same id so the nearby search will keep only each park's closest gate

parts.append(pd.DataFrame({
    'type': 'park', 'subtype': parks['function'].map(park_types), 'id': parks['id'],
    'name': parks['distName1'].fillna(parks['function']), 'info': parks['function'] + ' (' + parks['accessType'] + ' entrance)',
    'easting': parks.geometry.x, 'northing': parks.geometry.y,
}))

##### Listed Buildings

In [38]:
# Historic England listed buildings (via planning.data.gov.uk): living in or next to one
# limits what changes are allowed.

listed = gpd.read_file(raw/'planning'/'planning_listed_buildings.geojson').to_crs(bng)
listed = listed[listed['end-date'].isna() | (listed['end-date'] == '')]
listed_grade = 'Grade ' + listed['listed-building-grade'].fillna('unknown')

In [39]:
# Most listings are points; .centroid also turns the few outlines into a single centre point

centres = listed.geometry.centroid

In [40]:
parts.append(pd.DataFrame({
    'type': 'listed building', 'subtype': listed_grade, 'id': listed['reference'],
    'name': listed['name'], 'info': listed_grade + ' listed building',
    'easting': centres.x, 'northing': centres.y,
}))

In [41]:
print(parts[-1]['subtype'].value_counts())

subtype
Grade II     17193
Grade II*     1432
Grade I        602
Name: count, dtype: int64


In [42]:
# Stack every source into one table

points = pd.concat(parts, ignore_index=True)

In [43]:
# Whole metres are precise enough

points['easting'] = points['easting'].round().astype(int)
points['northing'] = points['northing'].round().astype(int)

In [44]:
# Keep only points inside London.

london_area = gpd.read_file(raw/'geography'/'boundaries_wards_2024.geojson').to_crs(bng).union_all()
xy = gpd.points_from_xy(points['easting'], points['northing'], crs=bng)
inside = xy.within(london_area)

In [45]:
print('Outside London:', points.loc[~inside, 'type'].value_counts().to_dict())

Outside London: {'listed building': 21, 'school': 5, 'park': 2}


In [46]:
points = points[inside].reset_index(drop=True)

In [47]:
# Latitude/longitude for drawing pins on the Streamlit map. Folium needs lat/lon, not metres

lonlat = gpd.points_from_xy(points['easting'], points['northing'], crs=bng).to_crs('EPSG:4326')
points['lon'] = lonlat.x.round(6)
points['lat'] = lonlat.y.round(6)

In [48]:
print(points.shape)
print(points['type'].value_counts())

(43588, 9)
type
listed building    19206
park               18121
school              3060
supermarket         2059
gp                  1142
Name: count, dtype: int64


In [49]:
print('Missing values:', points[['type', 'subtype', 'id', 'name', 'info']].isna().sum().to_dict())

Missing values: {'type': 0, 'subtype': 0, 'id': 0, 'name': 0, 'info': 0}


In [50]:
print(points[['lon', 'lat']].describe().loc[['min', 'max']])

          lon        lat
min -0.502512  51.292858
max  0.308955  51.688840


In [51]:
# Saving. The API will search this table for anything near a postcode

points.to_parquet(out_dir/'points.parquet', index=False)

In [52]:
points.groupby('type').head(1)[['type', 'subtype', 'name', 'info']]

,type,subtype,name,info
0,school,primary,The Aldgate School,"Primary, Voluntary aided school, ages 3-11, 22..."
3060,gp,gp practice,OAKLEIGH ROAD HEALTH CENTRE,"280 OAKLEIGH ROAD NORTH, WHETSTONE, LONDON, N2..."
4202,supermarket,Lidl,Lidl Alperton,"Lidl, 3,013 < 15,069 ft2 (280 < 1,400 m2)"
6261,park,playground,Play Space,Play Space (Pedestrian entrance)
24382,listed building,Grade II*,CRESCENT HOUSE INCLUDING GROUND FLOOR SHOPS AN...,Grade II* listed building
